# Milestone 4: what drives the crush, and implied against realised moves

This notebook imports `earnings_vol` and defines no logic of its own. The walkthrough is `docs/milestones/04-drivers.md`; the decisions are ADR 0005 §3-§5.

**The timing labels are provisional**: 20 press releases were looked up and every other event keeps the calendar's label until SEC EDGAR is set up. Every result below depends on them.

To rebuild (it needs the local cache; gate G1 runs before any H5-H7 result):

```
uv run python -m earnings_vol.panel.run
uv run python -m earnings_vol.stats.run --g1
uv run python -m earnings_vol.stats.run
uv run python tools/export_app_data.py
```

Contract schema version: `0.5.0`.

## 1. The surprise in price units, and its history (data model §3.11)
Made-up numbers. |SURP| = |reported - estimate| / the as-of price (the option forward times the discount factor at day -1). SUE_hist compares a quarter with the same company's previous ones: (this - mean of the previous) / their sd, with at least 4 of them.

In [ ]:
from datetime import date

from earnings_vol.panel import regressors

print("surprise_scaled", regressors.surprise_scaled(__import__("pandas").DataFrame(
    {"event_id": ["E1"], "surprise": [0.25]}), {"E1": 99.8}))
order = [("A", date(2025, 1, k), f"A{k}") for k in range(1, 7)]
vals = {"A1": 1.0, "A2": 2.0, "A3": 3.0, "A4": 4.0, "A5": 10.0, "A6": 5.0}
print("SUE_hist", {k: (round(v, 6), n) for k, (v, n) in regressors.sue_hist(order, vals, 8, 4).items()})


## 2. OLS with two-way clustered errors, and the Wald F (ADR 0005 §3)
Four made-up events, x = 0..3, y = 1, 3, 3, 8, on two dates and two firms. V = V_date + V_firm - V_date&firm, each CR1 with G/(G-1) (N-1)/(N-K); t on min(G) - 1 = 1 degree of freedom. The Wald test of (a, b) = (0, 1) is an F(2, 1), never a chi-square.

In [ ]:
import numpy as np

from earnings_vol.stats import regression

X = np.column_stack([np.ones(4), [0.0, 1.0, 2.0, 3.0]])
fit = regression.fit(np.array([1.0, 3.0, 3.0, 8.0]), X)
cov = regression.covariance(fit, X, ["d1", "d1", "d2", "d2"], ["f1", "f2", "f1", "f2"])
print("V", cov["V"].round(4).tolist(), "df", cov["df"])
b = regression.coef_test(fit, cov, 1)
print({k: round(v, 6) if isinstance(v, float) else v for k, v in b.items()})
w = regression.wald_test(fit, cov, np.eye(2), np.array([0.0, 1.0]))
print({k: round(v, 6) if isinstance(v, float) else v for k, v in w.items()})


## 3. H7's regressor in w units (ADR 0003 §c "Units")
The stored sigma_d is annualised: the ordinary variance per session is sigma_d^2 / 252. The AAPL worked example: E = 0.00088284, sigma_d = 0.2442.

In [ ]:
from earnings_vol.stats import drivers

print("E + sigma_d^2/252 =", round(float(drivers.implied_variance(np.array([0.00088284]), np.array([0.2442]))[0]), 7))
print("without the /252  =", round(0.00088284 + 0.2442 ** 2, 7))


## 4. Gate G1 for H5-H7, then the results (committed `results`)
G1 ran first. At the real design H5's, H6's and H7's primary statistics reject more than the 6.30% bar of 2,000 null replications: they are liberal, so a rejection by them would not be trusted. G1's size designs were declared for H1-H4 only (ADR 0005 §5), so H5-H7 stay in the declared Holm family H1-H7 with their real p-values (fix R1 after the milestone-4 review); none of them is rejected. H2's Holm p is printed with its G = 7 season-cut counterfactual (at this commit: H2 0.0469, G = 7 0.0781, not rejected; H5 0.0511). The `holm_post_gate` rows are a descriptive sensitivity: the Holm p over the family without the hypotheses whose size check failed (H1-H4: H2 0.0234, G = 7 0.0469).

In [ ]:
import pandas as pd

from earnings_vol import params, schema, store

derived = params.ROOT / "data" / "derived"
res = store.read_table(derived / "results.parquet", schema.RESULTS)
g1 = res[(res["family"] == "validation") & (res["measure"] == "g1:calibrated")
         & res["hypothesis"].isin(["H5", "H6", "H7"])]
print(g1[["hypothesis", "statistic", "role", "n", "estimate"]].to_string(index=False))
reg = res[(res["family"] == "regression") & res["hypothesis"].notna()]
cols = ["hypothesis", "role", "model_id", "adjustment", "term", "statistic", "estimate", "se", "stat",
        "df", "p_value"]
with pd.option_context("display.width", 180):
    print(reg[cols].to_string(index=False))
prim = res[res["family"].isin(["event_test", "regression"]) & (res["role"] == "primary")]
print(prim[["hypothesis", "statistic", "p_value", "p_holm"]].to_string(index=False))
cut = res[(res["subgroup_key"] == "season_cut") & (res["family"] == "event_test")]
print(cut[["hypothesis", "subgroup_value", "n_clusters", "p_value", "p_holm"]].to_string(index=False))
pg = res[res["measure"] == "holm_post_gate"]
print(pg[["hypothesis", "subgroup_key", "estimate"]].rename(columns={"estimate": "holm_post_gate"})
      .to_string(index=False))


## 5. The exploratory slopes, and the common IV factor
Every other named coefficient, with its BH q within the sample's regression rows. Each p and q comes from the same two-way t that G1 measured as liberal at this design for H5's and H6's coefficients (8.10% and 9.55% under its calibrated null); the other coefficients, SLOPE's included, were never sized by G1, so read them as indications, not tests at 5%. Then the milestone-3 open item: the crush baselines' rho-bar of 0.30 (Pynnonen's eq. 32), and what is left after a leave-one-firm-out factor is taken out, on made-up numbers (the real measurement is `research/prototypes/rhobar_crush.out.txt`).

In [ ]:
ex = res[(res["family"] == "regression") & (res["role"] == "exploratory") & res["p_value"].notna()]
with pd.option_context("display.width", 180, "display.max_rows", 80):
    print(ex[["model_id", "sample", "term", "estimate", "stat", "p_value", "q_bh"]].to_string(index=False))

from earnings_vol.stats import common_factor

rng = np.random.default_rng(0)
z = rng.standard_normal(200)[None, :] * 0.6 + rng.standard_normal((30, 200)) * 0.8
firms = [f"F{k}" for k in range(30)]
res_z, _ = common_factor.residualise(z, common_factor.loo_factor(z, firms), firms)
print("rho-bar", round(common_factor.rhobar(z), 3), "after the factor", round(common_factor.rhobar(res_z), 3))
